# Open two-level system with multiple observables

This example extends the minimal two-level model with radiative relaxation and pure dephasing. It demonstrates:

- GKSL collapse channels with rates stored separately from their operators;
- a stationary ground-state initial condition;
- a dissipatively broadened third-order rephasing response;
- polarization and action detection from the same third-order pathways;
- an action-population observable carrying its projection pulse;
- time-integrated radiative fluorescence after that projection pulse.

Natural units are used, with $\hbar=1$. Energies and rates are in eV, while times are in eV$^{-1}$.

In [ ]:
from pathlib import Path
import sys

import numpy as np

# Locate the project root when the notebook is launched from examples/
# or from another directory inside the repository.
current_directory = Path.cwd().resolve()
for candidate in (current_directory, *current_directory.parents):
    if (candidate / "projet_solver10.py").is_file():
        project_root = str(candidate)
        if project_root not in sys.path:
            sys.path.insert(0, project_root)
        break
else:
    raise RuntimeError("Could not locate the directory containing projet_solver10.py.")

from projet_solver10 import (
    EigenbasisKModel,
    ObservableSpec,
    SpectroscopyPlotter,
    SpectroscopySolver,
)
from projet_solver10.pathways import FrequencyPathway
from projet_solver10.protocols import standard_nq_protocol

## 1. Physical model

The basis is ordered as $\{|g\rangle, |e\rangle\}$. The Hamiltonian and transition operator are

$$
H = \omega_{eg}|e\rangle\langle e|,
\qquad
J = \mu\left(|e\rangle\langle g|+|g\rangle\langle e|\right).
$$

Radiative relaxation uses $C_{\mathrm{rad}}=|g\rangle\langle e|$. Pure dephasing uses $C_\phi=\sigma_z$. Because $\mathcal D[\sigma_z]\rho_{eg}=-2\rho_{eg}$, the collapse-channel rate is set to $\Gamma_\phi/2$ so that the optical coherence decays at the requested pure-dephasing rate $\Gamma_\phi$.

In [ ]:
# Model parameters
omega_eg = 2.0
dipole_strength = 0.5
gamma_radiative = 0.04
gamma_pure_dephasing = 0.03

# Bare matrices in the |g>, |e> basis
H_site = np.array(
    [[0.0, 0.0],
     [0.0, omega_eg]],
    dtype=complex,
)
interaction_site = dipole_strength * np.array(
    [[0.0, 1.0],
     [1.0, 0.0]],
    dtype=complex,
)

ground_population_op = np.array([[1.0, 0.0], [0.0, 0.0]], dtype=complex)
excited_population_op = np.array([[0.0, 0.0], [0.0, 1.0]], dtype=complex)
radiative_lowering = np.array([[0.0, 1.0], [0.0, 0.0]], dtype=complex)
dephasing_operator = np.array([[-1.0, 0.0], [0.0, 1.0]], dtype=complex)

print("Hamiltonian:\n", H_site)
print("Interaction operator:\n", interaction_site)
print("Radiative collapse operator:\n", radiative_lowering)

## 2. Build the model and solver

`c_ops_raw` contains `(operator, rate)` pairs. The rate must not be folded into the operator. `EigenbasisKModel` names these channels according to their input order, so `c_op_0` is radiative relaxation and `c_op_1` is pure dephasing in this example.

In [ ]:
model = EigenbasisKModel(
    H_site,
    interaction_site,
    c_ops_raw=(
        (radiative_lowering, gamma_radiative),
        (dephasing_operator, gamma_pure_dephasing / 2.0),
    ),
    observable_op_arrays={
        "polarization": interaction_site,
        "ground_population": ground_population_op,
        "excited_population": excited_population_op,
    },
)

solver = SpectroscopySolver(backend="dense", eta=0.002)
solver.feed_model(model)

print(solver.summary())
print("Jump channels:", solver.jump_channel_names())
print("Memory estimate:", solver.estimate_memory())

## 3. Verify the stationary initial state

At zero temperature, the supplied ground state should satisfy $\mathcal L_{\mathrm{eff}}\rho_g=0$. This check also verifies that the collapse channels are compatible with the claimed initial state.

In [ ]:
rho_ground = ground_population_op.copy()
rho_ground_vector = rho_ground.reshape(-1, order="F")
stationarity_residual = np.linalg.norm(
    solver.backend.generator.matvec(rho_ground_vector)
)

print(f"Stationarity residual: {stationarity_residual:.3e}")
assert stationarity_residual < 1e-12

## 4. Dissipatively broadened third-order rephasing response

For a two-level system, the rephasing $\chi^{(3)}$ signal contains the ground-state-bleach and stimulated-emission pathways

$$R_1=(B_u,K_u,B_d),\qquad R_2=(B_u,B_d,K_u).$$

Both pathways carry the coherence history $q=(-1,0,+1)$. There is no excited-state-absorption pathway because the model contains no doubly excited manifold. Radiative relaxation contributes $\Gamma_1/2$ to the optical-coherence decay, while pure dephasing contributes $\Gamma_\phi$.

In [ ]:
pathway_r1 = FrequencyPathway(
    name="R1",
    interactions=("Bu", "Ku", "Bd"),
    component="rephasing",
    detection="polarization",
)
pathway_r2 = FrequencyPathway(
    name="R2",
    interactions=("Bu", "Bd", "Ku"),
    component="rephasing",
    detection="polarization",
)
protocol_1q = standard_nq_protocol(
    order=1,
    nq_interval=1,
    detection_interval=3,
    n_interactions=3,
    nq_axis="omega_1q",
    detection_axis="omega_emit",
)
omega_1q = np.linspace(-2.5, -1.5, 101)
omega_emit = np.linspace(1.5, 2.5, 101)
population_waiting_time = 10.0

rephasing_result = solver.generate_spectrum(
    protocol_1q,
    axes={"omega_1q": omega_1q, "omega_emit": omega_emit},
    fixed_coordinates={"t2": population_waiting_time},
    pathways=(pathway_r1, pathway_r2),
)
rephasing_signal = rephasing_result.components["rephasing"]

assert pathway_r1.coherence_orders == (-1, 0, 1)
assert pathway_r2.coherence_orders == (-1, 0, 1)
assert np.all(np.isfinite(rephasing_signal))
assert np.max(np.abs(rephasing_signal)) > 0.0

peak_index = np.unravel_index(
    np.argmax(np.abs(rephasing_signal)), rephasing_signal.shape
)
print("Strongest rephasing coordinate:")
print(f"  omega_1q   = {omega_1q[peak_index[0]]:.4f} eV")
print(f"  omega_emit = {omega_emit[peak_index[1]]:.4f} eV")

In [ ]:
plotter = SpectroscopyPlotter(detection_phase=0.0)
plotter.plot_spectrum_result(
    rephasing_result,
    params={
        "source": "components",
        "names": ["rephasing"],
        "view": "all",
        "normalization": "row",
        "title": r"Open two-level $\chi^{(3)}$ rephasing response",
        "labels": (
            r"Emission energy $\omega_{\mathrm{emit}}$ (eV)",
            r"Excitation energy $\omega_{1Q}$ (eV)",
        ),
        "diagonals": "auto",
        "style": {
            "cmap": "RdYlBu_r",
            "abs_cmap": "magma",
            "levels": 30,
            "contour_lines": False,
        },
        "show": True,
    },
)

## 5. Polarization and action detection from the same $\chi^{(3)}$ pathways

The solver first propagates the same three field interactions used for the third-order rephasing polarization. Polarization is contracted directly from that response, whereas each action observable carries an optional fourth interaction that projects the final optical coherence onto the population sector $q=0$. The common three-pulse response is therefore propagated only once and then reused by the coherent and action-detection branches.

For both rephasing pathways, the projection interaction is `Bu`, which changes the final order $q=+1$ to $q=0$. The pathways and protocol nevertheless remain third order: the projection pulse belongs to the observable, not to `FrequencyPathway.interactions`.

In [ ]:
pathways = (pathway_r1, pathway_r2)
assert all(len(pathway.interactions) == 3 for pathway in pathways)
print("Third-order pathway coherence orders:")
for pathway in pathways:
    print(f"  {pathway.name}: {pathway.coherence_orders}")

In [ ]:
radiative_channel = solver.jump_channel_names()[0]
integration_window = 5.0 / gamma_radiative

observables = {
    "polarization": "polarization",
    "action_population": ObservableSpec.action(
        "action_population",
        fourth_interaction="Bu",
        operator="excited_population",
    ),
    "action_fluorescence": ObservableSpec.mean_jump(
        "action_fluorescence",
        radiative_channel,
        time_window=(0.0, integration_window),
        fourth_interaction="Bu",
    ),
}

omega_1q = np.linspace(-2.5, -1.5, 41)
omega_emit = np.linspace(1.5, 2.5, 41)
result = solver.generate_spectrum(
    protocol_1q,
    {"omega_1q": omega_1q, "omega_emit": omega_emit},
    fixed_coordinates={"t2": population_waiting_time},
    pathways=pathways,
    observables=observables,
)

## 6. Validate the three observable branches

All three signals originate from the same $\chi^{(3)}$ propagation. Only `action_population` and `action_fluorescence` apply `Bu` before detection; the fluorescence is then integrated over the finite post-projection window.

In [ ]:
for pathway in pathways:
    name = pathway.name
    polarization = result.observables["polarization"][name]
    action_population = result.observables["action_population"][name]
    action_fluorescence = result.observables["action_fluorescence"][name]

    for signal in (polarization, action_population, action_fluorescence):
        assert np.all(np.isfinite(signal))
        assert np.max(np.abs(signal)) > 0.0

    print(f"{name}: max polarization = {np.max(np.abs(polarization)):.6e}")
    print(f"{name}: max action population = {np.max(np.abs(action_population)):.6e}")
    print(f"{name}: max action fluorescence = {np.max(np.abs(action_fluorescence)):.6e}")

print("Polarization and action detection use the same chi(3) pathways.")

In [ ]:
for observable_name, title in (
    ("polarization", r"$\chi^{(3)}$ polarization"),
    ("action_population", "Action-detected excited population"),
    ("action_fluorescence", "Action-detected integrated fluorescence"),
):
    plotter.plot_spectrum_result(
        result,
        params={
            "source": "observables",
            "observable": observable_name,
            "pathways": ["R1", "R2"],
            "view": "all",
            "normalization": "row",
            "title": title,
            "labels": (
                r"Emission energy $\omega_{\mathrm{emit}}$ (eV)",
                r"Excitation energy $\omega_{1Q}$ (eV)",
            ),
            "diagonals": "auto",
            "show": True,
        },
    )

## What this example shows

1. Collapse operators and rates are supplied separately.
2. The ground state is stationary under the field-free GKSL generator.
3. Radiative relaxation and pure dephasing broaden a third-order rephasing spectrum.
4. Action observables carry the `Bu` projection pulse while every pathway remains third order.
5. One spectrum calculation returns polarization, action population, and integrated action fluorescence from the same propagated response.